                    TRANSFORMER
                         │
                         ↓
                    Embeddings
                         ↓
              Positional Information
                         ↓
              ┌────────────────────┐
              │ Transformer Block  │
              │                    │
              │ Multi-Head         │
              │ Self-Attention     │
              │       ↓            │
              │ Residual + Norm    │
              │       ↓            │
              │ Feed Forward       │
              │       ↓            │
              │ Residual + Norm    │
              └─────────┬──────────┘
                        ↓
                 Repeat N times
                        ↓
                   Final Layer
                        ↓
                 Next-token output

In [ ]:
text = """
i love cats
i love dogs
i like cats
i like dogs
cats are cute
dogs are cute
cats are animals
dogs are animals
i love animals
"""

# Tokenization

In [ ]:
tokens = text.lower().split()

print(tokens)

['i', 'love', 'cats', 'i', 'love', 'dogs', 'i', 'like', 'cats', 'i', 'like', 'dogs', 'cats', 'are', 'cute', 'dogs', 'are', 'cute', 'cats', 'are', 'animals', 'dogs', 'are', 'animals', 'i', 'love', 'animals']


# Vocabulary

In [ ]:
vocab = sorted(set(tokens))

print(vocab)

['animals', 'are', 'cats', 'cute', 'dogs', 'i', 'like', 'love']


# ID Mapping

In [ ]:
stoi = {word: i for i, word in enumerate(vocab)}
itos = {i: word for word, i in stoi.items()}

print(stoi)
print(itos)

{'animals': 0, 'are': 1, 'cats': 2, 'cute': 3, 'dogs': 4, 'i': 5, 'like': 6, 'love': 7}
{0: 'animals', 1: 'are', 2: 'cats', 3: 'cute', 4: 'dogs', 5: 'i', 6: 'like', 7: 'love'}


# Convert our entire Dataset to Id's

In [ ]:
encoded = [stoi[word] for word in tokens]

print(encoded)
print(len(encoded))

[5, 7, 2, 5, 7, 4, 5, 6, 2, 5, 6, 4, 2, 1, 3, 4, 1, 3, 2, 1, 0, 4, 1, 0, 5, 7, 0]
27


# Training Sequences

In [ ]:
seq_length = 4

X = []
y = []

for i in range(len(encoded) - seq_length):
    X.append(encoded[i:i + seq_length])
    y.append(encoded[i + 1:i + seq_length + 1])

# Convert to tensors

In [ ]:
import torch

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(X.shape)
print(y.shape)

torch.Size([23, 4])
torch.Size([23, 4])


# **Embedding Layer**

**Token IDs → Embeddings**

In [ ]:
import torch
import torch.nn as nn

vocab_size = len(vocab)
embedding_dim = 8

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

In [ ]:
print(embedding.weight)

Parameter containing:
tensor([[-5.7441e-01, -1.4589e+00, -1.1538e+00, -9.7095e-01, -8.3378e-01,
         -3.3228e-01,  3.6442e-02, -2.5096e-01],
        [ 3.0766e-02, -1.1643e-01,  1.0502e+00, -2.8196e-01,  1.5765e+00,
         -4.4141e-01, -3.6717e-02,  8.2094e-01],
        [-1.0712e+00, -1.5249e+00, -1.5469e+00, -9.6422e-02, -1.0696e+00,
          1.2302e+00,  1.2588e-01, -2.0545e+00],
        [ 9.0664e-01,  1.7510e+00, -8.9940e-01, -2.8975e-01, -9.7535e-01,
          4.3730e-02,  3.3931e-01,  7.2546e-01],
        [ 1.4438e+00,  9.3076e-02, -9.6488e-01,  1.6523e-01,  1.5041e+00,
          2.6113e-05,  7.6293e-01,  3.9237e-01],
        [ 1.7570e+00, -1.4326e+00, -2.9293e-01, -2.4475e-01, -4.5343e-01,
          6.4688e-01,  2.5835e-01,  1.8593e-01],
        [ 5.6881e-01, -6.5070e-01, -1.6947e+00, -6.1506e-01, -7.9407e-01,
          4.0271e-01, -8.9004e-02, -4.1275e-01],
        [-1.3215e+00, -1.4911e+00,  5.7796e-01, -1.5604e+00, -2.4211e+00,
          3.5372e-01, -1.0351e-01,  2.5868e

In [ ]:
x = torch.tensor([[5, 7, 2, 5]])

embedded = embedding(x)
print(embedded)

print(x.shape)
print(embedded.shape)

tensor([[[ 1.7570, -1.4326, -0.2929, -0.2447, -0.4534,  0.6469,  0.2583,
           0.1859],
         [-1.3215, -1.4911,  0.5780, -1.5604, -2.4211,  0.3537, -0.1035,
           0.2587],
         [-1.0712, -1.5249, -1.5469, -0.0964, -1.0696,  1.2302,  0.1259,
          -2.0545],
         [ 1.7570, -1.4326, -0.2929, -0.2447, -0.4534,  0.6469,  0.2583,
           0.1859]]], grad_fn=<EmbeddingBackward0>)
torch.Size([1, 4])
torch.Size([1, 4, 8])


**Positional Embedding**

In [ ]:
max_seq_length = 4

position_embedding = nn.Embedding(
    max_seq_length,
    embedding_dim
)

In [ ]:
positions = torch.arange(4)

print(positions)

tensor([0, 1, 2, 3])


In [ ]:
pos_emb = position_embedding(positions)

print(pos_emb.shape)

torch.Size([4, 8])


**Add token + position embeddings**

In [ ]:
x_emb = embedded + pos_emb

# **Self Attention**

In [ ]:
import torch
import torch.nn as nn

attention_dim = embedding_dim

Wq = nn.Linear(
    embedding_dim,
    attention_dim
)

Wk = nn.Linear(
    embedding_dim,
    attention_dim
)

Wv = nn.Linear(
    embedding_dim,
    attention_dim
)

Q = Wq(x_emb)

K = Wk(x_emb)

V = Wv(x_emb)


print("\nQ shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)


Q shape: torch.Size([1, 4, 8])
K shape: torch.Size([1, 4, 8])
V shape: torch.Size([1, 4, 8])


In [ ]:
scores = torch.matmul(
    Q,
    K.transpose(-2, -1)
)

print("\nAttention scores shape:")
print(scores.shape)

print("\nRaw attention scores:")
print(scores)



Attention scores shape:
torch.Size([1, 4, 4])

Raw attention scores:
tensor([[[-0.9642,  0.1076,  0.1640,  0.1954],
         [ 1.6308, -1.7842, -2.1998, -0.8413],
         [-0.1732, -0.7324, -0.8732, -0.1175],
         [-0.0378,  0.9114, -0.5293,  0.5149]]], grad_fn=<UnsafeViewBackward0>)


**Scaling**

In [ ]:
d_k = K.size(-1)

scores = scores / (d_k ** 0.5)

print("\nScaled attention scores:")
print(scores)


Scaled attention scores:
tensor([[[-0.3409,  0.0380,  0.0580,  0.0691],
         [ 0.5766, -0.6308, -0.7778, -0.2974],
         [-0.0613, -0.2589, -0.3087, -0.0416],
         [-0.0134,  0.3222, -0.1871,  0.1821]]], grad_fn=<DivBackward0>)


**Casual Mask**

In [ ]:
mask = torch.tril(
    torch.ones(
        seq_length,
        seq_length
    )
)

print("\nCausal mask:")
print(mask)


Causal mask:
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])


In [ ]:
scores = scores.masked_fill(
    mask == 0,
    float("-inf")
)

print("\nMasked scores:")
print(scores)


Masked scores:
tensor([[[-0.3409,    -inf,    -inf,    -inf],
         [ 0.5766, -0.6308,    -inf,    -inf],
         [-0.0613, -0.2589, -0.3087,    -inf],
         [-0.0134,  0.3222, -0.1871,  0.1821]]], grad_fn=<MaskedFillBackward0>)


**Softmax**

In [ ]:
import torch.nn.functional as F

attention_weights = F.softmax(
    scores,
    dim=-1
)

print("\nAttention weights:")
print(attention_weights)

print("\nAttention weights shape:")
print(attention_weights.shape)


Attention weights:
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.7698, 0.2302, 0.0000, 0.0000],
         [0.3844, 0.3155, 0.3001, 0.0000],
         [0.2245, 0.3140, 0.1887, 0.2729]]], grad_fn=<SoftmaxBackward0>)

Attention weights shape:
torch.Size([1, 4, 4])


**Multiply Attention Weight x Values**

In [ ]:
attention_output = torch.matmul(
    attention_weights,
    V
)

print("\nSelf-attention output:")
print(attention_output)

print("\nSelf-attention output shape:")
print(attention_output.shape)


Self-attention output:
tensor([[[-3.7960e-01, -3.8479e-01,  4.0820e-01, -1.8534e-01, -3.4408e-01,
          -1.2046e+00,  5.9462e-01,  6.5754e-02],
         [-4.5776e-01, -5.6682e-01,  5.8874e-01, -5.4983e-01, -1.1728e-01,
          -6.3082e-01,  2.9084e-01, -1.2874e-01],
         [-5.8253e-01, -8.1712e-01,  8.8008e-01, -8.9436e-01,  1.7690e-04,
           3.5086e-01, -1.3160e-01, -4.0151e-01],
         [-6.1663e-01, -5.5803e-01,  6.5797e-01, -5.8019e-01, -8.5832e-02,
           2.6021e-01, -1.5248e-01, -9.3038e-02]]],
       grad_fn=<UnsafeViewBackward0>)

Self-attention output shape:
torch.Size([1, 4, 8])


In [ ]:
print("\n" + "=" * 60)
print("FINAL SELF-ATTENTION PIPELINE")
print("=" * 60)

print("Input token IDs:")
print(x)

print("\nQ:")
print(Q)

print("\nK:")
print(K)

print("\nV:")
print(V)

print("\nAttention weights:")
print(attention_weights)

print("\nFinal self-attention output:")
print(attention_output)


FINAL SELF-ATTENTION PIPELINE
Input token IDs:
tensor([[5, 7, 2, 5]])

Q:
tensor([[[ 0.9194, -0.3967, -1.2306,  0.2809, -0.3347,  0.4851, -0.1927,
          -0.4571],
         [-1.1676,  0.6635,  1.4889,  0.3375, -0.8478,  0.0928,  0.8929,
           1.2857],
         [ 0.2973,  0.1961, -0.3424,  0.3669, -0.9447, -0.4924,  0.0684,
           0.2019],
         [ 0.9472, -0.5340, -1.2479, -0.6489, -0.6748,  0.0197, -0.2398,
          -0.2359]]], grad_fn=<ViewBackward0>)

K:
tensor([[[-0.2984,  0.7674, -0.3142, -1.0242,  0.1091,  0.1999,  0.3648,
           1.0382],
         [ 0.0577,  0.4893, -1.3175, -1.1352,  1.3316, -0.4590,  1.1123,
           0.3745],
         [-0.6211,  0.4095, -1.0009,  0.2103,  1.3467, -0.4129,  0.3632,
          -0.7170],
         [-0.3770, -0.7546, -0.4564, -0.1052,  0.1921, -0.3898,  0.0914,
           0.0403]]], grad_fn=<ViewBackward0>)

V:
tensor([[[-0.3796, -0.3848,  0.4082, -0.1853, -0.3441, -1.2046,  0.5946,
           0.0658],
         [-0.7192, -1.1756

# **Multi-Head Self-Attention**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super().__init__()

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads

        # Each head gets part of the embedding dimensions
        self.head_dim = embedding_dim // num_heads

        assert embedding_dim % num_heads == 0, \
            "embedding_dim must be divisible by num_heads"

        # Q, K, V projections
        self.Wq = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.Wk = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.Wv = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        # Final projection after combining heads
        self.out_proj = nn.Linear(
            embedding_dim,
            embedding_dim
        )


    def forward(self, x):

        batch_size, seq_length, embedding_dim = x.shape

        # ------------------------------------------------
        # 1. Create Q, K, V
        # ------------------------------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)


        # ------------------------------------------------
        # 2. Split into multiple heads
        # ------------------------------------------------

        Q = Q.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        )

        K = K.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        )

        V = V.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        )


        # ------------------------------------------------
        # 3. Move heads before sequence dimension
        # ------------------------------------------------

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)


        # ------------------------------------------------
        # 4. Attention scores
        # ------------------------------------------------

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )


        # ------------------------------------------------
        # 5. Scale
        # ------------------------------------------------

        scores = scores / (self.head_dim ** 0.5)


        # ------------------------------------------------
        # 6. Causal mask
        # ------------------------------------------------

        mask = torch.tril(
            torch.ones(
                seq_length,
                seq_length,
                device=x.device
            )
        )

        scores = scores.masked_fill(
            mask == 0,
            float("-inf")
        )


        # ------------------------------------------------
        # 7. Softmax
        # ------------------------------------------------

        attention_weights = F.softmax(
            scores,
            dim=-1
        )


        # ------------------------------------------------
        # 8. Weighted sum of Values
        # ------------------------------------------------

        attention_output = torch.matmul(
            attention_weights,
            V
        )


        # ------------------------------------------------
        # 9. Combine heads
        # ------------------------------------------------

        attention_output = attention_output.transpose(1, 2)

        attention_output = attention_output.contiguous().view(
            batch_size,
            seq_length,
            self.embedding_dim
        )


        # ------------------------------------------------
        # 10. Final projection
        # ------------------------------------------------

        output = self.out_proj(
            attention_output
        )

        return output

# Residual Connection

In [ ]:
class AttentionWithResidualNorm(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super().__init__()

        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads
        )

        self.norm = nn.LayerNorm(
            embedding_dim
        )


    def forward(self, x):

        # Keep original input
        residual = x

        # Self-attention
        attention_output = self.attention(x)

        # Residual connection
        x = residual + attention_output

        # Layer normalization
        x = self.norm(x)

        return x

In [ ]:
embedding_dim = 8
num_heads = 2

attention_block = AttentionWithResidualNorm(
    embedding_dim,
    num_heads
)

output = attention_block(x_emb)

print("Input shape:")
print(x_emb.shape)

print("\nOutput shape:")
print(output.shape)

Input shape:
torch.Size([1, 4, 8])

Output shape:
torch.Size([1, 4, 8])


# Feed Forward Neural Network

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, embedding_dim):

        super().__init__()

        self.network = nn.Sequential(

            # Expand
            nn.Linear(
                embedding_dim,
                embedding_dim * 4
            ),

            # Non-linearity
            nn.GELU(),

            # Compress back
            nn.Linear(
                embedding_dim * 4,
                embedding_dim
            )
        )


    def forward(self, x):

        return self.network(x)

In [ ]:
ffn = FeedForward(embedding_dim)

ffn_output = ffn(output)

print("Input to FFN:")
print(output.shape)

print("\nOutput from FFN:")
print(ffn_output.shape)

Input to FFN:
torch.Size([1, 4, 8])

Output from FFN:
torch.Size([1, 4, 8])


# Transformer Block

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super().__init__()

        # Multi-head self-attention
        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads
        )

        # Feed-forward network
        self.ffn = FeedForward(
            embedding_dim
        )

        # Layer normalizations
        self.norm1 = nn.LayerNorm(
            embedding_dim
        )

        self.norm2 = nn.LayerNorm(
            embedding_dim
        )


    def forward(self, x):

        # ==========================================
        # 1. Self-Attention
        # ==========================================

        attention_output = self.attention(x)


        # ==========================================
        # 2. Residual + LayerNorm
        # ==========================================

        x = self.norm1(
            x + attention_output
        )


        # ==========================================
        # 3. Feed Forward
        # ==========================================

        ffn_output = self.ffn(x)


        # ==========================================
        # 4. Residual + LayerNorm
        # ==========================================

        x = self.norm2(
            x + ffn_output
        )


        return x

In [ ]:
embedding_dim = 8
num_heads = 2

transformer_block = TransformerBlock(
    embedding_dim,
    num_heads
)

output = transformer_block(x_emb)

print("Input:")
print(x_emb.shape)

print("\nOutput:")
print(output.shape)

Input:
torch.Size([1, 4, 8])

Output:
torch.Size([1, 4, 8])


# Stacking Multiple Layers

In [ ]:
class TransformerStack(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        num_layers
    ):

        super().__init__()

        self.blocks = nn.ModuleList([

            TransformerBlock(
                embedding_dim,
                num_heads
            )

            for _ in range(num_layers)
        ])


    def forward(self, x):

        for block in self.blocks:

            x = block(x)

        return x

In [ ]:
lm_head = nn.Linear(
    embedding_dim,
    vocab_size
)

logits = lm_head(output)

**Softmax**

In [ ]:
probabilities = F.softmax(
    logits,
    dim=-1
)

# Tiny GPT

In [ ]:
class TinyGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_heads,
        num_layers,
        max_seq_length
    ):

        super().__init__()

        # -----------------------------
        # Token Embedding
        # -----------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # -----------------------------
        # Positional Embedding
        # -----------------------------

        self.position_embedding = nn.Embedding(
            max_seq_length,
            embedding_dim
        )

        # -----------------------------
        # Transformer Blocks
        # -----------------------------

        self.blocks = nn.ModuleList([

            TransformerBlock(
                embedding_dim,
                num_heads
            )

            for _ in range(num_layers)

        ])

        # -----------------------------
        # Final LayerNorm
        # -----------------------------

        self.final_norm = nn.LayerNorm(
            embedding_dim
        )

        # -----------------------------
        # Language Model Head
        # -----------------------------

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )


    def forward(self, x):

        batch_size, seq_length = x.shape

        # -----------------------------
        # Token embeddings
        # -----------------------------

        token_emb = self.token_embedding(x)

        # -----------------------------
        # Position embeddings
        # -----------------------------

        positions = torch.arange(
            seq_length,
            device=x.device
        )

        pos_emb = self.position_embedding(
            positions
        )

        # -----------------------------
        # Combine embeddings
        # -----------------------------

        x = token_emb + pos_emb

        # -----------------------------
        # Transformer blocks
        # -----------------------------

        for block in self.blocks:

            x = block(x)

        # -----------------------------
        # Final normalization
        # -----------------------------

        x = self.final_norm(x)

        # -----------------------------
        # Language model head
        # -----------------------------

        logits = self.lm_head(x)

        return logits

**Create the Model**

In [ ]:
vocab_size = len(vocab)

embedding_dim = 8

num_heads = 2

num_layers = 4

max_seq_length = 4

model = TinyGPT(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    max_seq_length=max_seq_length
)

print(model)

TinyGPT(
  (token_embedding): Embedding(8, 8)
  (position_embedding): Embedding(4, 8)
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (attention): MultiHeadAttention(
        (Wq): Linear(in_features=8, out_features=8, bias=True)
        (Wk): Linear(in_features=8, out_features=8, bias=True)
        (Wv): Linear(in_features=8, out_features=8, bias=True)
        (out_proj): Linear(in_features=8, out_features=8, bias=True)
      )
      (ffn): FeedForward(
        (network): Sequential(
          (0): Linear(in_features=8, out_features=32, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=32, out_features=8, bias=True)
        )
      )
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
    )
  )
  (final_norm): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=8, out_features=8, bias=True)
)


In [ ]:
sample = X[0].unsqueeze(0)

print("Sample:")
print(sample)

print("Shape:")
print(sample.shape)

logits = model(sample)

print("Logits shape:")
print(logits.shape)

Sample:
tensor([[5, 7, 2, 5]])
Shape:
torch.Size([1, 4])
Logits shape:
torch.Size([1, 4, 8])


In [ ]:
# Get the logits for the last token position
last_logits = logits[:, -1, :]

print("Last token logits:")
print(last_logits)

print("\nShape:")
print(last_logits.shape)

Last token logits:
tensor([[ 0.0659,  0.6691, -0.1518, -0.9738, -0.3699, -0.5232,  0.8846,  0.1189]],
       grad_fn=<SelectBackward0>)

Shape:
torch.Size([1, 8])


In [ ]:
predicted_id = torch.argmax(
    last_logits,
    dim=-1
)

print("Predicted token ID:")
print(predicted_id)

predicted_word = itos[predicted_id.item()]

print("\nPredicted word:")
print(predicted_word)

Predicted token ID:
tensor([6])

Predicted word:
like


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 500

for epoch in range(epochs):

    # Forward pass
    logits = model(X)

    # Reshape logits
    logits = logits.view(
        -1,
        vocab_size
    )

    # Reshape targets
    targets = y.view(
        -1
    )

    # Calculate loss
    loss = criterion(
        logits,
        targets
    )

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update model weights
    optimizer.step()

    # Print progress
    if (epoch + 1) % 50 == 0:

        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 50/500, Loss: 1.4625
Epoch 100/500, Loss: 1.0376
Epoch 150/500, Loss: 0.7614
Epoch 200/500, Loss: 0.5978
Epoch 250/500, Loss: 0.4929
Epoch 300/500, Loss: 0.4247
Epoch 350/500, Loss: 0.3870
Epoch 400/500, Loss: 0.3627
Epoch 450/500, Loss: 0.3459
Epoch 500/500, Loss: 0.3355


In [ ]:
# model.eval()

# while True:

#     sentence = input("\nEnter your sentence (type 'exit' to quit): ")

#     if sentence.lower() == "exit":
#         break

#     # Convert sentence into tokens
#     words = sentence.lower().split()

#     # Check vocabulary
#     unknown_words = [word for word in words if word not in stoi]

#     if unknown_words:
#         print("Unknown words:", unknown_words)
#         print("Please use only these words:")
#         print(list(stoi.keys()))
#         continue

#     # Convert words → token IDs
#     token_ids = [stoi[word] for word in words]

#     # Convert to tensor
#     input_tensor = torch.tensor(
#         [token_ids],
#         dtype=torch.long
#     )

#     # Check sequence length
#     if input_tensor.size(1) > max_seq_length:
#         print(f"Please enter at most {max_seq_length} words.")
#         continue

#     # Prediction
#     with torch.no_grad():

#         logits = model(input_tensor)

#         # Take the last token's prediction
#         last_logits = logits[:, -1, :]

#         # Get highest-scoring token
#         predicted_id = torch.argmax(
#             last_logits,
#             dim=-1
#         ).item()

#     # Convert ID → word
#     predicted_word = itos[predicted_id]

#     print("Input:", sentence)
#     print("Predicted next word:", predicted_word)

In [ ]:
def generate(
    model,
    input_text,
    max_new_tokens,
    temperature=1.0
):

    model.eval()

    words = input_text.lower().split()

    token_ids = [stoi[word] for word in words]

    tokens = torch.tensor(
        [token_ids],
        dtype=torch.long
    )

    for _ in range(max_new_tokens):

        tokens_for_model = tokens[:, -max_seq_length:]

        with torch.no_grad():

            logits = model(tokens_for_model)

        # Last token's logits
        last_logits = logits[:, -1, :]

        # Apply temperature
        last_logits = last_logits / temperature

        # Convert logits to probabilities
        probabilities = torch.softmax(
            last_logits,
            dim=-1
        )

        # Randomly sample according to probabilities
        predicted_id = torch.multinomial(
            probabilities,
            num_samples=1
        ).item()

        # Add predicted token
        new_token = torch.tensor(
            [[predicted_id]],
            dtype=torch.long
        )

        tokens = torch.cat(
            [tokens, new_token],
            dim=1
        )

    result = [
        itos[token_id.item()]
        for token_id in tokens[0]
    ]

    return " ".join(result)

# Temperatures in transformers

**Test Temperature = 1**

In [ ]:
print(
    generate(
        model,
        "i love",
        max_new_tokens=3,
        temperature=1.0
    )
)

i love dogs i like


**Try Low Temperature**

In [ ]:
print(
    generate(
        model,
        "i love",
        max_new_tokens=3,
        temperature=0.3
    )
)

i love dogs i like


**Try High Temperature**

In [ ]:
print(
    generate(
        model,
        "i",
        max_new_tokens=3,
        temperature=2.0
    )
)

i love cute dogs


# K Sampling

In [ ]:
def generate(
    model,
    input_text,
    max_new_tokens,
    temperature=1.0,
    top_k=None
):

    model.eval()

    words = input_text.lower().split()

    token_ids = [stoi[word] for word in words]

    tokens = torch.tensor(
        [token_ids],
        dtype=torch.long
    )

    for _ in range(max_new_tokens):

        tokens_for_model = tokens[:, -max_seq_length:]

        with torch.no_grad():
            logits = model(tokens_for_model)

        # Last token's logits
        last_logits = logits[:, -1, :]

        # Temperature
        last_logits = last_logits / temperature

        # -------------------------
        # TOP-K FILTERING
        # -------------------------

        if top_k is not None:

            values, indices = torch.topk(
                last_logits,
                k=top_k,
                dim=-1
            )

            filtered_logits = torch.full_like(
                last_logits,
                float("-inf")
            )

            filtered_logits.scatter_(
                -1,
                indices,
                values
            )

            last_logits = filtered_logits

        # Convert to probabilities
        probabilities = torch.softmax(
            last_logits,
            dim=-1
        )

        # Sample
        predicted_id = torch.multinomial(
            probabilities,
            num_samples=1
        ).item()

        # Add predicted token
        new_token = torch.tensor(
            [[predicted_id]],
            dtype=torch.long
        )

        tokens = torch.cat(
            [tokens, new_token],
            dim=1
        )

    result = [
        itos[token_id.item()]
        for token_id in tokens[0]
    ]

    return " ".join(result)

In [ ]:
print(
    generate(
        model,
        "i",
        max_new_tokens=3,
        temperature=1.0,
        top_k=3
    )
)

i love dogs i


# Top P Nucleus Filtering

In [ ]:
def generate(
    model,
    input_text,
    max_new_tokens,
    temperature=1.0,
    top_p=0.9
):

    model.eval()

    words = input_text.lower().split()

    token_ids = [stoi[word] for word in words]

    tokens = torch.tensor(
        [token_ids],
        dtype=torch.long
    )

    for _ in range(max_new_tokens):

        tokens_for_model = tokens[:, -max_seq_length:]

        with torch.no_grad():
            logits = model(tokens_for_model)

        last_logits = logits[:, -1, :]

        # Temperature
        last_logits = last_logits / temperature

        # Convert to probabilities
        probs = torch.softmax(last_logits, dim=-1)

        # Sort probabilities
        sorted_probs, sorted_indices = torch.sort(
            probs,
            descending=True,
            dim=-1
        )

        # Cumulative probability
        cumulative_probs = torch.cumsum(
            sorted_probs,
            dim=-1
        )

        # Remove tokens after top_p
        mask = cumulative_probs > top_p

        # Shift mask right
        mask[..., 1:] = mask[..., :-1].clone()
        mask[..., 0] = False

        # Set removed probabilities to zero
        sorted_probs[mask] = 0

        # Normalize again
        sorted_probs = sorted_probs / sorted_probs.sum(
            dim=-1,
            keepdim=True
        )

        # Sample
        sampled_index = torch.multinomial(
            sorted_probs,
            num_samples=1
        )

        predicted_id = sorted_indices.gather(
            -1,
            sampled_index
        ).item()

        # Add token
        new_token = torch.tensor(
            [[predicted_id]],
            dtype=torch.long
        )

        tokens = torch.cat(
            [tokens, new_token],
            dim=1
        )

    result = [
        itos[token.item()]
        for token in tokens[0]
    ]

    return " ".join(result)

In [ ]:
print(
    generate(
        model,
        "i love",
        max_new_tokens=4,
        temperature=0.8,
        top_p=0.9
    )
)

i love dogs i like cats
